# Invoice Extraction — Problem & Solution (Colab)

This notebook mirrors the **problem statement** and **solution** used in this project, and runs the same pipeline in Google Colab.

## 1. Problem statement

**Input:** Invoice images (e.g. `batch1-0348.jpg`, `batch1-0349.jpg`).

**Goal:** Extract the same **structured fields** from every image:

| Field | Description |
|-------|-------------|
| `seller_name` | Vendor/seller name |
| `seller_tax_id` | Seller tax ID (VAT, NIP, etc.) |
| `client_name` | Buyer/client name |
| `client_tax_id` | Client tax ID |
| `invoice_number` | Invoice number |
| `invoice_date` | Invoice date |
| `net_worth` | Net amount |
| `vat` | VAT amount |
| `gross_worth` | Gross total |

**Output:**
- One reconciled row per image in `output.csv`
- Field-by-field Pipeline A vs B comparison in `comparison_report.csv`
- Summary metrics: field accuracy, document accuracy

## 2. Solution overview

The project uses **two extraction pipelines** and **reconciles** their results:

- **Pipeline A:** OCR (Tesseract) → **LLM** (Gemini / OpenAI / Ollama) → structured JSON
- **Pipeline B:** OCR (Tesseract) → **regex/heuristics** → same structured fields

For each image:
1. Run both pipelines → get two dicts (A and B)
2. **Normalize** both (dates → ISO, numbers → float, names cleaned)
3. **Compare** A vs B per field → build comparison report
4. **Reconcile**: when A and B disagree, prefer Pipeline B (configurable)
5. Write one row to `output.csv` and field rows to `comparison_report.csv`

**LLM (Pipeline A) in Colab:**
- **Gemini (recommended, free):** `INVOICE_LLM_PROVIDER=gemini` + `GEMINI_API_KEY` from [Google AI Studio](https://aistudio.google.com/apikey)
- **OpenAI:** `INVOICE_LLM_PROVIDER=openai` + `OPENAI_API_KEY`

**Images:** Default `invoice_extractor/images/`. Names must match `{PREFIX}{index:04d}.{EXT}` (e.g. `batch1-0348.jpg`). Use `INVOICE_IMAGES_DIR`, `INVOICE_IMAGE_START`, `INVOICE_IMAGE_END` to configure.

## 3. Data flow (per image)

```
Image path
    → Pipeline A: OCR → LLM → dict (REQUIRED_FIELDS)
    → Pipeline B: OCR → regex → dict (REQUIRED_FIELDS)
    → normalize_invoice_dict() on both
    → compare_invoices() → rows for comparison_report.csv
    → build_reconciled_invoice() → one row for output.csv
```

## 4. Clone repo and install dependencies

If you **uploaded** the project (ZIP or folder) instead of cloning, skip the clone cell and run:

```python
# %cd /content/invoice-extraction   # or your uploaded folder path
```

In [ ]:
# Clone the repo (replace with your repo URL if different)
!git clone https://github.com/YOUR_USERNAME/invoice-extraction.git
%cd invoice-extraction

In [ ]:
# Install Tesseract OCR (Linux) and Python deps (run from repo root)
!apt-get update -qq && apt-get install -y -qq tesseract-ocr
!pip install -q -r invoice_extractor/requirements.txt

## 5. Set API key and run options

In [ ]:
import os

# --- LLM: Gemini (free) or OpenAI ---
# Gemini (recommended in Colab): get API key at https://aistudio.google.com/apikey
os.environ["INVOICE_LLM_PROVIDER"] = "gemini"
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    pass
# Or set directly (avoid committing real keys): os.environ["GEMINI_API_KEY"] = "your-key"

# Alternative: use OpenAI instead
# os.environ["INVOICE_LLM_PROVIDER"] = "openai"
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")  # or set directly

# Optional: point to images on Google Drive
# from google.colab import drive
# drive.mount("/content/drive")
# os.environ["INVOICE_IMAGES_DIR"] = "/content/drive/MyDrive/invoice_images"

# Optional: image range and naming (default prefix batch1-, range 348–350, ext jpg)
# os.environ["INVOICE_IMAGE_PREFIX"] = "batch1-"
# os.environ["INVOICE_IMAGE_START"] = "348"
# os.environ["INVOICE_IMAGE_END"] = "350"
# os.environ["INVOICE_IMAGE_EXT"] = "jpg"

## 6. Run the pipeline

In [ ]:
# Run from repo root (after clone: we're already in invoice-extraction)
!python -m invoice_extractor.main

## 7. Download or view outputs

In [ ]:
from google.colab import files

# List outputs
!ls -la invoice_extractor/outputs/

# Download CSVs (uncomment to use)
# files.download("invoice_extractor/outputs/output.csv")
# files.download("invoice_extractor/outputs/comparison_report.csv")